In [1]:
# Task 1  —  Design a Class
# Objective: Design a class with a constructor, attributes and methods
# Design a Python class called TrainingRun that represents a single model-training run.
# Requirements:
# 1.	Add a constructor (__init__) that accepts model_name and learning_rate, and sets sensible instance attributes.
# 2.	Add a class attribute run_count that tracks how many TrainingRun instances have been created across the whole program.
# 3.	Add at least two instance methods: one that starts the run (e.g. start()) and one that reports status (e.g. summary()).
# 4.	Add one @classmethod, e.g. from_config(cls, config_dict), that builds a TrainingRun from a dictionary.
# Acceptance criteria
# Creating three TrainingRun instances should leave TrainingRun.run_count == 3. Each instance's own attributes (model_name, learning_rate) must
# remain independent per object.
# Stretch goal: add a @staticmethod that validates a learning rate is within a sane range (e.g. 0 < lr < 1) and call it from __init__.

class TrainingRun:
    run_count = 0

    def __init__(self, model_name, learning_rate):
        if not self.validate_learning_rate(learning_rate):
            raise ValueError("Learning rate must be between 0 and 1")

        self.model_name = model_name
        self.learning_rate = learning_rate
        self.status = "Not Started"

        TrainingRun.run_count += 1

    def start(self):
        self.status = "Running"
        print(f"{self.model_name} training started")

    def summary(self):
        print(
            f"Model: {self.model_name}, "
            f"Learning Rate: {self.learning_rate}, "
            f"Status: {self.status}"
        )

    @classmethod
    def from_config(cls, config_dict):
        return cls(
            config_dict["model_name"],
            config_dict["learning_rate"]
        )

    @staticmethod
    def validate_learning_rate(lr):
        return 0 < lr < 1


run1 = TrainingRun("ResNet", 0.01)
run2 = TrainingRun("BERT", 0.001)
run3 = TrainingRun.from_config({
    "model_name": "GPT",
    "learning_rate": 0.0001
})

run1.start()

run1.summary()
run2.summary()
run3.summary()

print("Run Count:", TrainingRun.run_count)

ResNet training started
Model: ResNet, Learning Rate: 0.01, Status: Running
Model: BERT, Learning Rate: 0.001, Status: Not Started
Model: GPT, Learning Rate: 0.0001, Status: Not Started
Run Count: 3


In [2]:
# Task 2  —  Encapsulation
# PILLAR: ENCAPSULATION
# Objective: Apply Python's encapsulation conventions (public / _protected / __private)
# Extend TrainingRun with an internal “private” field and a “protected” status field.
# 1.	Add a name-mangled attribute __api_key inside __init__ (e.g. a fake token string).
# 2.	Add a _status attribute, then expose it through a @property called status with a matching setter.
# 3.	The status setter should only allow the values "pending", "running", or "done" — raise ValueError on anything else.
# 4.	Demonstrate (e.g. in a comment or a short script) that __api_key is not directly accessible from outside the class as instance.__api_key,
# but is still reachable via name-mangling as instance._TrainingRun__api_key.
# Acceptance criteria
# run.status = "paused" raises ValueError. Accessing run.__api_key from outside the class raises AttributeError, while run._TrainingRun__api_key
# still succeeds.
# Stretch goal: add a short written note (2–3 sentences) contrasting this with Java's compiler-enforced private keyword — what exactly does Python trust the developer to do here?

class TrainingRun:
    run_count = 0

    def __init__(self, model_name, learning_rate):
        self.model_name = model_name
        self.learning_rate = learning_rate

        self.__api_key = "abc123token"
        self._status = "pending"

        TrainingRun.run_count += 1

    @property
    def status(self):
        return self._status

    @status.setter
    def status(self, value):
        valid_statuses = {"pending", "running", "done"}

        if value not in valid_statuses:
            raise ValueError(
                "Status must be 'pending', 'running', or 'done'"
            )

        self._status = value


run = TrainingRun("ResNet", 0.01)

print(run.status)

run.status = "running"
print(run.status)

try:
    run.status = "paused"
except ValueError as e:
    print(e)

try:
    print(run.__api_key)
except AttributeError as e:
    print(e)

print(run._TrainingRun__api_key)

pending
running
Status must be 'pending', 'running', or 'done'
'TrainingRun' object has no attribute '__api_key'
abc123token


In [3]:
# Task 3  —  Inheritance
# PILLAR: INHERITANCE
# Objective: Implement inheritance and method overriding
# Create a subclass, LRSchedulerRun(TrainingRun), representing a training run whose learning rate changes over time.
# 1.	Call super().__init__(...) to reuse the parent constructor, then add an extra attribute, e.g. schedule (a list of learning rates per epoch).
# 2.	Override summary() so it prints the current scheduled learning rate in addition to whatever the parent's summary() shows
# — call the parent version with super().summary() rather than duplicating its logic.
# 3.	Verify that an LRSchedulerRun still increments the shared TrainingRun.run_count class attribute defined in Task 1.
# Acceptance criteria
# LRSchedulerRun().summary() must include both the parent class's original output and the new schedule information, produced via super(),
# not copy-pasted logic.

class TrainingRun:
    run_count = 0

    def __init__(self, model_name, learning_rate):
        self.model_name = model_name
        self.learning_rate = learning_rate
        TrainingRun.run_count += 1

    def summary(self):
        print(f"Model: {self.model_name}")
        print(f"Learning Rate: {self.learning_rate}")


class LRSchedulerRun(TrainingRun):

    def __init__(self, model_name, learning_rate, schedule):
        super().__init__(model_name, learning_rate)
        self.schedule = schedule

    def summary(self):
        super().summary()
        print(f"Schedule: {self.schedule}")
        print(f"Current LR: {self.schedule[0]}")


run1 = TrainingRun("ResNet", 0.01)

run2 = LRSchedulerRun(
    "BERT",
    0.001,
    [0.001, 0.0005, 0.0001]
)

run2.summary()

print("Run Count:", TrainingRun.run_count)

Model: BERT
Learning Rate: 0.001
Schedule: [0.001, 0.0005, 0.0001]
Current LR: 0.001
Run Count: 2


In [4]:
# Task 4  —  Polymorphism
# PILLAR: POLYMORPHISM
# Objective: Use polymorphism to treat different subclasses interchangeably
# Create a second subclass, EarlyStoppingRun(TrainingRun), that overrides summary() differently (e.g. reporting a patience attribute
# — how many epochs to wait before stopping).
# 1.	Write a function print_all_summaries(runs) that loops over a list containing a MIX of TrainingRun, LRSchedulerRun, and EarlyStoppingRun
# instances, calling .summary() on each one.
# 2.	The function must NOT check each object's type (no isinstance() or type() branching) — the whole point is that the same .summary() 
# call produces different, correct behavior depending on each object's actual class.
# 3.	Build a list containing at least one instance of each of the three classes and pass it to print_all_summaries to prove it works.
# Acceptance criteria
# print_all_summaries(...) correctly calls each subclass's own overridden summary() with zero type-checking logic inside the function itself.

# Discussion prompt for the room: this is polymorphism in action — the caller doesn't need to know or care which concrete subclass it's holding. 
# Ask trainees where they've seen this pattern already in Java (e.g. a List<Shape> calling .area() on each element).

class TrainingRun:
    def __init__(self, model_name, learning_rate):
        self.model_name = model_name
        self.learning_rate = learning_rate

    def summary(self):
        print(f"Model: {self.model_name}")
        print(f"Learning Rate: {self.learning_rate}")


class LRSchedulerRun(TrainingRun):
    def __init__(self, model_name, learning_rate, schedule):
        super().__init__(model_name, learning_rate)
        self.schedule = schedule

    def summary(self):
        super().summary()
        print(f"Schedule: {self.schedule}")


class EarlyStoppingRun(TrainingRun):
    def __init__(self, model_name, learning_rate, patience):
        super().__init__(model_name, learning_rate)
        self.patience = patience

    def summary(self):
        super().summary()
        print(f"Patience: {self.patience}")


def print_all_summaries(runs):
    for run in runs:
        run.summary()
        print()


run1 = TrainingRun("ResNet", 0.01)

run2 = LRSchedulerRun(
    "BERT",
    0.001,
    [0.001, 0.0005, 0.0001]
)

run3 = EarlyStoppingRun(
    "GPT",
    0.0001,
    5
)

runs = [run1, run2, run3]

print_all_summaries(runs)

Model: ResNet
Learning Rate: 0.01

Model: BERT
Learning Rate: 0.001
Schedule: [0.001, 0.0005, 0.0001]

Model: GPT
Learning Rate: 0.0001
Patience: 5



In [5]:
# Task 5  —  Duck Typing vs. Interfaces
# PILLAR: CONTRACTS
# Objective: Contrast duck typing / ABCs with Java's interfaces
# Take the following scenario: several “pipeline step” classes (e.g. Cleaner, Tokenizer, Normalizer) each need a process(data) method.
# 1.	First, implement it the duck-typed way: three unrelated classes that each simply define process(self, data), with no shared base class.
# Write a run_pipeline(steps, data) function that calls .process() on each step in turn.
# 2.	Second, re-implement the same three classes so they inherit from an abstract base class Step(ABC) with an @abstractmethod
# process(self, data). Show that Step() itself cannot be instantiated.
# 3.	Write two or three sentences (as a code comment or short note) on when you would reach for each approach on a real team.
# Acceptance criteria
# Both versions of run_pipeline must work correctly. Attempting to instantiate the ABC version's Step() directly must raise TypeError.

from abc import ABC, abstractmethod

# Duck Typing

class Cleaner:
    def process(self, data):
        return data.strip()

class Tokenizer:
    def process(self, data):
        return data.split()

class Normalizer:
    def process(self, data):
        return [word.lower() for word in data]

def run_pipeline(steps, data):
    for step in steps:
        data = step.process(data)
    return data


steps = [Cleaner(), Tokenizer(), Normalizer()]

print(run_pipeline(steps, "  Hello PYTHON World  "))


# ABC (Abstract Base Class)

class Step(ABC):

    @abstractmethod
    def process(self, data):
        pass


class CleanerStep(Step):
    def process(self, data):
        return data.strip()


class TokenizerStep(Step):
    def process(self, data):
        return data.split()


class NormalizerStep(Step):
    def process(self, data):
        return [word.lower() for word in data]


def run_pipeline_abc(steps, data):
    for step in steps:
        data = step.process(data)
    return data


steps2 = [
    CleanerStep(),
    TokenizerStep(),
    NormalizerStep()
]

print(run_pipeline_abc(steps2, "  Hello PYTHON World  "))


try:
    step = Step()
except TypeError as e:
    print(e)


# Notes:
# Duck typing is great for small projects and quick development because
# any object with a process() method can participate in the pipeline.
#
# ABCs are useful in larger teams and codebases because they enforce a
# clear contract. Every subclass must implement process(), making the
# code easier to understand, validate, and maintain.

['hello', 'python', 'world']
['hello', 'python', 'world']
Can't instantiate abstract class Step without an implementation for abstract method 'process'


In [6]:
# Task 6  —  Dunder Methods
# PILLAR: DUNDER METHODS
# Objective: Implement at least one dunder method
# Go back to your TrainingRun class from Task 1 and implement __str__ so that print(my_run) produces a clean, human-readable summary.
# 1.	Implement __str__(self) returning something like "TrainingRun(model='gpt-mini', lr=0.01)".
# 2.	Implement __eq__(self, other) so that two TrainingRun instances are considered equal if they have the same model_name and learning_rate.
# Acceptance criteria
# print(run) must not show the default <TrainingRun object at 0x...> representation. Two separately-constructed runs with 
# identical model_name and learning_rate must compare equal with ==.

# Stretch goal: also implement __repr__ and explain, in a comment, the difference in intent between __str__ and __repr__.

class TrainingRun:
    def __init__(self, model_name, learning_rate):
        self.model_name = model_name
        self.learning_rate = learning_rate

    def __str__(self):
        """Human-readable representation."""
        return f"TrainingRun(model='{self.model_name}', lr={self.learning_rate})"

    def __eq__(self, other):
        """Two runs are equal if model_name and learning_rate match."""
        if not isinstance(other, TrainingRun):
            return NotImplemented
        return (
            self.model_name == other.model_name
            and self.learning_rate == other.learning_rate
        )

    def __repr__(self):
        """
        Developer-oriented representation.
        __str__ is meant for end users and readability.
        __repr__ is meant for developers and debugging, and ideally
        should be detailed enough to recreate the object.
        """
        return (
            f"TrainingRun(model_name='{self.model_name}', "
            f"learning_rate={self.learning_rate})"
        )


# Example usage
run1 = TrainingRun("gpt-mini", 0.01)
run2 = TrainingRun("gpt-mini", 0.01)
run3 = TrainingRun("gpt-large", 0.01)

print(run1)          # Uses __str__
print(run1 == run2)  # True
print(run1 == run3)  # False
print(repr(run1))    # Uses __repr__

TrainingRun(model='gpt-mini', lr=0.01)
True
False
TrainingRun(model_name='gpt-mini', learning_rate=0.01)
